# 05 — Recalage signalé : sub-146

Le contrôle qualité du recalage rigide T1 → FLAIR (étape 3.2, J-031) a signalé **un seul sujet sur 170** : sub-146 (Amsterdam GE 3T, cas de test). Corrélation avec la T1 alignée par les organisateurs (elastix) 0,905 (seuil 0,95), écart géométrique jusqu'à 3,3 mm, rotation relative 1,2°.

Question : **est-ce notre recalage (ANTs) qui a échoué, ou celui des organisateurs (elastix) ?** On ne dispose d'aucune vérité terrain sur l'alignement : on juge en confrontant chaque T1 alignée à la **FLAIR**, l'image de référence (fixe), avec des critères indépendants des deux outils.

| Section | Contenu |
|---|---|
| 1 | Chiffres du contrôle qualité, sub-146 face aux autres sujets du même scanner |
| 2 | Superpositions : contours de la FLAIR sur la T1 alignée par ANTs et par elastix |
| 3 | Damiers FLAIR / T1 |
| 4 | Juge indépendant : information mutuelle avec la FLAIR, dans le cerveau |
| 5 | Où les deux transformations diffèrent-elles ? (carte du déplacement) |
| 6 | Un recalage relancé autrement donne-t-il le même résultat ? |
| 7 | Avis |

Un sujet de référence du même scanner (`REF`, recalage non signalé) sert de comparaison.

In [ ]:
import ants
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
import SimpleITK as sitk
from sklearn.metrics import mutual_info_score

from wmh_multisite.config import load_config, resolve_path
from wmh_multisite.preproc import PIPELINE
from wmh_multisite.preproc.register import compare_transforms, elastix_euler, head_points, subject_paths
from wmh_multisite.utils import io

SUB, REF = "sub-146", "sub-114"     # REF: median subject of the same scanner (Amsterdam GE 3T)
cfg = load_config()
D = resolve_path(cfg, "derivatives") / PIPELINE
part = pd.read_csv(resolve_path(cfg, "bids") / "participants.tsv", sep="\t").set_index("participant_id")
report = pd.read_csv(D / "registration_report.csv")
report["scanner"] = report.participant_id.map(part.scanner)

## 1. Les chiffres du contrôle qualité

In [ ]:
same = report[report.scanner == part.loc[SUB, "scanner"]]
print(same[["corr_vs_elastix", "disp_mean_mm", "disp_max_mm", "rot_diff_deg"]].describe().loc[["min", "50%", "max"]].round(3))
same.sort_values("corr_vs_elastix").head(5)[["participant_id", "corr_vs_elastix", "disp_mean_mm", "disp_max_mm", "rot_diff_deg", "flagged"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, col in zip(axes, ["corr_vs_elastix", "disp_max_mm", "rot_diff_deg"]):
    ax.hist(report[col], bins=40, color="0.7")
    ax.axvline(report.loc[report.participant_id == SUB, col].iloc[0], color="red", label=SUB)
    ax.set_title(col); ax.legend()
plt.suptitle("170 sujets : sub-146 en rouge"); plt.tight_layout(); plt.show()

## 2. Superpositions : les contours de la FLAIR sur chaque T1 alignée

On calcule, sur la grille FLAIR :
- **T1 alignée par ANTs** : notre transformation appliquée à la T1 brute ;
- **T1 alignée par elastix** : `orig/T1` des organisateurs.

Puis on trace les **contours de la FLAIR** (bords des ventricules et du cerveau, détectés sur la FLAIR) par-dessus chaque T1. Si l'alignement est bon, les contours tombent sur les mêmes structures de la T1.

In [ ]:
def load_case(sub):
    p = subject_paths(cfg, sub)
    flair = ants.image_read(str(p["flair_raw"]), pixeltype="float")
    t1 = ants.image_read(str(p["t1_raw"]), pixeltype="float")
    ants_t1 = ants.apply_transforms(flair, t1, [str(p["xfm"])], interpolator="linear").numpy()
    elastix_t1 = io.load_data(p["t1_elastix"])
    brain = io.load_labels(D / sub / "anat" / f"{sub}_space-FLAIR_desc-brain_mask.nii.gz", allowed=(0, 1)).astype(bool)
    return {"p": p, "flair": flair.numpy(), "ants": ants_t1, "elastix": elastix_t1, "brain": brain,
            "affine": io.load(p["flair_raw"]).affine, "flair_img": flair, "t1_img": t1}

def canon(vol, affine):
    img = nib.as_closest_canonical(nib.Nifti1Image(vol.astype(np.float32), affine))
    return img.get_fdata(dtype=np.float32), img.header.get_zooms()[:3]

def edges(vol, brain):
    # FLAIR contours inside the brain: strong gradient magnitude (ventricles, cortex, brain border)
    g = np.linalg.norm(np.gradient(vol), axis=0) * brain
    return g > np.percentile(g[brain], 90)

S, R = load_case(SUB), load_case(REF)

def show_overlays(case, title, ks=(0.35, 0.5, 0.65)):
    fl, z = canon(case["flair"], case["affine"])
    ed, _ = canon(edges(case["flair"], case["brain"]).astype(np.float32), case["affine"])
    fig, axes = plt.subplots(len(ks), 3, figsize=(13, 4.1 * len(ks)), squeeze=False)
    for r, frac in enumerate(ks):
        k = int(frac * (fl.shape[2] - 1))
        for c, (name, vol) in enumerate([("FLAIR", case["flair"]), ("T1 alignée par ANTs", case["ants"]),
                                          ("T1 alignée par elastix", case["elastix"])]):
            v, _ = canon(vol, case["affine"])
            ax = axes[r, c]
            ax.imshow(v[:, :, k].T, origin="lower", cmap="gray", vmin=0, vmax=np.percentile(v[v > 0], 99.5), aspect=z[1] / z[0])
            if c > 0:
                ax.contour(ed[:, :, k].T, levels=[0.5], colors=["red"], linewidths=0.5)
            ax.set_title(f"{name} — coupe {k}", fontsize=8); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

show_overlays(S, f"{SUB} : contours de la FLAIR (rouge) sur chaque T1 alignée")

In [ ]:
show_overlays(R, f"{REF} (référence, non signalé)", ks=(0.5,))

## 3. Damiers FLAIR / T1

Les cases alternent la FLAIR et la T1 (intensités normalisées) : aux bords des cases, les structures (ventricules, scissure médiane, contour du cerveau) doivent se prolonger sans cassure.

In [ ]:
def checker(a, b, brain, n=8):
    a = a / np.percentile(a[brain], 99); b = b / np.percentile(b[brain], 99)
    ii, jj = np.indices(a.shape[:2])
    board = ((ii // max(a.shape[0] // n, 1) + jj // max(a.shape[1] // n, 1)) % 2).astype(bool)
    return np.where(board[..., None], a, b)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))
for r, (name, key) in enumerate([("ANTs", "ants"), ("elastix", "elastix")]):
    cb = checker(S["flair"], S[key], S["brain"])
    v, z = canon(cb, S["affine"])
    for c, frac in enumerate((0.35, 0.5, 0.65)):
        k = int(frac * (v.shape[2] - 1))
        axes[r, c].imshow(v[:, :, k].T, origin="lower", cmap="gray", vmin=0, vmax=1.1, aspect=z[1] / z[0])
        axes[r, c].set_title(f"{SUB} — FLAIR / T1 {name}, coupe {k}", fontsize=8); axes[r, c].axis("off")
plt.tight_layout(); plt.show()

## 4. Un juge indépendant : l'information mutuelle avec la FLAIR

La corrélation du contrôle qualité compare notre T1 à **celle d'elastix** : elle dit que les deux diffèrent, pas laquelle est juste. On mesure donc, pour chaque T1 alignée, sa ressemblance avec la **FLAIR** elle-même, dans le cerveau, par l'information mutuelle (le critère du recalage, ici calculé sur tous les voxels du cerveau, histogramme de 32 classes). La meilleure des deux est la plus haute.

In [ ]:
def mi(a, b, mask, bins=32):
    qa = np.digitize(a[mask], np.quantile(a[mask], np.linspace(0, 1, bins + 1)[1:-1]))
    qb = np.digitize(b[mask], np.quantile(b[mask], np.linspace(0, 1, bins + 1)[1:-1]))
    return mutual_info_score(qa, qb)

rows = []
for sub, case in [(SUB, S), (REF, R)]:
    m = case["brain"] & (case["ants"] > 0) & (case["elastix"] > 0)
    rows.append({"sujet": sub, "MI FLAIR / T1 ANTs": mi(case["flair"], case["ants"], m),
                 "MI FLAIR / T1 elastix": mi(case["flair"], case["elastix"], m),
                 "corr. T1 ANTs / T1 elastix": np.corrcoef(case["ants"][m], case["elastix"][m])[0, 1]})
mi_table = pd.DataFrame(rows).round(4)
mi_table

## 5. Où les deux transformations diffèrent-elles ?

Pour chaque voxel du cerveau, distance (mm) entre le point où ANTs va lire la T1 et le point où elastix va la lire. Une rotation produit un écart qui **croît avec la distance au centre de rotation** ; une translation, un écart uniforme.

In [ ]:
p = S["p"]
ours = sitk.ReadTransform(str(p["xfm"])).Downcast()
ref = elastix_euler(p["elastix_params"])
fl = S["flair_img"]
idx = np.argwhere(S["brain"])
pts = np.asarray(fl.origin) + (idx * np.asarray(fl.spacing)) @ np.asarray(fl.direction).T
disp = np.array([np.linalg.norm(np.subtract(ours.TransformPoint(tuple(x)), ref.TransformPoint(tuple(x)))) for x in pts])
dmap = np.zeros(S["brain"].shape, np.float32); dmap[tuple(idx.T)] = disp
print(f"déplacement dans le cerveau : médiane {np.median(disp):.2f} mm, p95 {np.percentile(disp, 95):.2f} mm, max {disp.max():.2f} mm")
v, z = canon(dmap, S["affine"])
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, frac in zip(axes, (0.35, 0.5, 0.65)):
    k = int(frac * (v.shape[2] - 1))
    im = ax.imshow(np.ma.masked_where(v[:, :, k] == 0, v[:, :, k]).T, origin="lower", cmap="magma", vmin=0, vmax=3.5, aspect=z[1] / z[0])
    ax.set_title(f"coupe {k}", fontsize=8); ax.axis("off")
fig.colorbar(im, ax=axes, label="écart ANTs / elastix (mm)"); plt.show()

## 6. Un recalage relancé autrement donne-t-il le même résultat ?

Si notre recalage était tombé dans un mauvais optimum local, un recalage partant d'une autre initialisation devrait trouver autre chose. On relance ANTs (même critère) en partant de la transformation d'elastix, et on compare la solution obtenue à nos deux candidats.

In [ ]:
init = sitk.Euler3DTransform(ref)
init_file = str(resolve_path(cfg, "derivatives") / PIPELINE / "_tmp_init_sub146.mat")
sitk.WriteTransform(init, init_file)
ants.config._random_seed = cfg["preproc"]["rigid"]["seed"]
reg = ants.registration(fixed=S["flair_img"], moving=S["t1_img"], type_of_transform="Rigid", initial_transform=[init_file])
alt = sitk.ReadTransform(reg["fwdtransforms"][0]).Downcast()
sample = head_points(S["flair_img"], S["brain"])
alt_t1 = ants.apply_transforms(S["flair_img"], S["t1_img"], reg["fwdtransforms"], interpolator="linear").numpy()
m = S["brain"] & (alt_t1 > 0) & (S["ants"] > 0) & (S["elastix"] > 0)
pd.DataFrame([
    {"comparaison": "relancé (init. elastix) vs ANTs initial", **compare_transforms(alt, ours, sample)},
    {"comparaison": "relancé (init. elastix) vs elastix", **compare_transforms(alt, ref, sample)},
]).assign(**{"MI FLAIR / T1 relancée": round(mi(S["flair"], alt_t1, m), 4)})

In [ ]:
import os
os.remove(init_file)

## 7. Avis (rédigé à partir des sorties de l'exécution du 2026-10-04)

**Constats**
1. sub-146 est le seul sujet signalé sur 170, et il est très loin des autres : corrélation avec elastix 0,905 (2e plus faible : 0,974), écart médian de **2,0 mm** dans le cerveau (max 3,3 mm), rotation relative 1,2°. L'écart est réparti sur tout le cerveau (rotation + translation), pas localisé.
2. **Juge indépendant (section 4)** : l'information mutuelle avec la FLAIR, dans le cerveau, vaut **0,245 pour la T1 d'elastix contre 0,162 pour la nôtre**. Sur le sujet de référence, les deux sont équivalentes (0,289 et 0,297). Le recalage des organisateurs est donc nettement meilleur pour ce sujet.
3. **Recalage relancé à partir de la solution d'elastix (section 6)** : ANTs s'en éloigne de 1,5 mm en moyenne et aboutit à une solution intermédiaire (MI 0,19). Le critère d'ANTs (calculé sur tout le volume, cou et fond compris, avec échantillonnage de 20 % des voxels) a donc un optimum différent de celui qui aligne le cerveau : ce n'est pas un simple mauvais départ.
4. **Essai complémentaire, hors notebook** : le même recalage restreint au cerveau de la T1 (`moving_mask` = masque HD-BET) échoue franchement (20 mm d'écart), alors qu'il fonctionne sur le sujet de référence. 
5. **La FLAIR de sub-146 est atypique** (sections 2 et 3) : très bruitée, avec un contraste inhabituel (cortex clair, substance blanche peu différenciée). Le lien statistique entre ses intensités et celles de la T1 est faible, ce qui rend l'information mutuelle peu discriminante : c'est la cause probable.

**Avis.** Notre recalage de sub-146 est **probablement décalé d'environ 2 mm** ; celui d'elastix est meilleur. Pour ce sujet, **utiliser la transformation d'elastix** (exception documentée, transformation fournie avec le jeu de données) plutôt que de régler ANTs sur un cas unique. Et signaler la **FLAIR atypique** au contrôle qualité (Phase 4) : l'image elle-même, pas seulement son recalage, est hors norme, et la segmentation de ce sujet devra être interprétée avec prudence.